> **The performance cliff.** You spent weeks implementing FlashAttention's tiling algorithm in Python — the math is right, the output matches PyTorch's reference, and you proved that keeping tiles in SRAM beats reading from HBM. Then you run a wall-clock comparison.
>
> Your Python FlashAttention: **47ms per forward pass.**
> PyTorch's `scaled_dot_product_attention`: **2.1ms per forward pass.** That's 22× slower.
>
> The bottleneck is not your algorithm — it's the language. Python interpreter overhead, object allocation, and loop overhead dominate. Every iteration crosses the Python/C boundary hundreds of times. The tiling insight is correct; the execution engine is wrong.
>
> **Triton closes the gap.** It lets you write GPU kernels in Python-like syntax that compile directly to CUDA PTX — the same low-level instructions cuBLAS and FlashAttention-2 generate. You already know the algorithm from Chapter 4. This chapter teaches you the language it actually runs in, starting from a 5-line vector addition and building to a fused softmax that matches FlashAttention-2's bandwidth utilization.

# Custom GPU Kernels with Triton

A Python FlashAttention is 22× slower than PyTorch's SDPA. The algorithm is right; the execution layer is wrong. This notebook builds from a 5-line vector addition to a fused softmax kernel — the same pattern that gives FlashAttention-2 87% of peak HBM bandwidth.

| Part | Kernel | What it proves | Why right now? |
|------|--------|----------------|----------------|
| 1 | Vector addition | Triton programming model: grids, blocks, pointers | Without `tl.load/store` and `program_id`, you cannot read *any* Triton kernel — learn the vocabulary first |
| 2 | Tiled matmul | Block-level parallelism; compare to torch.matmul | Tiling is the key move in FlashAttention — see it in Triton before applying it to softmax in Part 4 |
| 3 | Fused GELU+bias | Fusion eliminates HBM roundtrips; same result, less bandwidth | Quantifies *why* we bother: 50% fewer HBM accesses = 2× more bandwidth for the same operation |
| 4 | Tiled softmax | FlashAttention's inner loop in Triton; verify vs. reference | This is the kernel that makes FlashAttention-2 reach 87% of peak HBM bandwidth |
| 5 | Autotuning | `@triton.autotune` selects optimal block size for your GPU | A kernel fast on A100 can be slow on RTX 4090 — block size is hardware-specific, not theory-derivable |
| 6 | Toy → real | Where are Triton kernels in `torch.compile`'d code? | Closes the loop: you rarely write Triton by hand, but now you can *read* and *debug* what the compiler generates |

---

## Prerequisite Bridge — From Ch1 and Ch4

| Foundation | Role in this notebook |
|---|---|
| CUDA grid/block/thread model (Ch1) | Triton's launch grid creates program instances; the compiler maps each program onto CUDA threads and warps |
| Warp occupancy (Ch1) | Tile shape and `num_warps` jointly determine register pressure, resident warps, and occupancy |
| Tiling insight (Ch4) | Triton's tile-level operations implement exactly the tiling from Ch4 |
| Online softmax (Ch4) | Part 4's fused tiled softmax is a Triton building block of the Ch4 algorithm |

> **If Triton is not installed:** All kernel cells show annotated pseudocode AND a PyTorch reference implementation. The concepts fully transfer — only the compilation step requires a CUDA GPU.

In [ ]:
import subprocess, sys
# Install torch/numpy/matplotlib only if missing
for pkg in ['torch', 'numpy', 'matplotlib']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HAS_GPU = torch.cuda.is_available()

#  Try to import Triton
TRITON_AVAILABLE = False
try:
    import triton
    import triton.language as tl
    TRITON_AVAILABLE = True
    print(f" Triton {triton.__version__} available — kernels will compile and run")
except ImportError:
    print("Triton not installed — showing annotated pseudocode + PyTorch reference for each kernel")
    print("To install: pip install triton  (requires CUDA GPU)")

print(f"\nDevice: {DEVICE}")
print(f"GPU available: {HAS_GPU}")
print()

# Running example dimensions (same as Ch1 and Ch4)
B, S, D = 8, 128, 64
print(f"Running example: (B={B}, S={S}, D={D}) attention head (same as Ch1 and Ch4)")
print(f"  This is the workload we're progressively porting from Python to Triton")


---

## Part 1 — Vector Addition: The "Hello World" of Triton

**Why do you need this now?** Every subsequent kernel — tiled matmul, fused GELU, softmax — uses the same three primitives: `tl.program_id` to identify the current program instance, `tl.load` to pull data from slow HBM into fast registers, and `tl.store` to write results back. If you don't internalize these three operations on a trivial example (`C[i] = A[i] + B[i]`), the 60-line softmax kernel in Part 4 will be opaque. Start here.

Vector addition is the simplest GPU kernel: for each element i, `C[i] = A[i] + B[i]`. In CUDA, you'd write a C++ kernel function with `__global__` and explicit `blockIdx`/`threadIdx` math. In Triton, you write Python with decorators and tile-level operators — the compiler handles warp scheduling, memory coalescing, and CUDA PTX generation.

**Key Triton concepts introduced here:**
- `@triton.jit` — decorates a Python function to be compiled as a GPU kernel
- `tl.load(ptr + offsets, mask=mask)` — load a *tile* of elements from GPU HBM into registers
- `tl.store(ptr + offsets, values, mask=mask)` — write a tile of results back to HBM
- `tl.program_id(axis=0)` — which program instance in the launch grid is this? (conceptually like `blockIdx.x` in CUDA)

One Triton **program instance** processes one logical tile (often called a block in Triton tutorials). The grid launches `ceil(N / BLOCK_SIZE)` program instances. A program is not a CUDA thread: Triton's compiler maps its vector operations onto a cooperating group of CUDA threads organized into warps. Multiple programs may be resident on one SM when registers and shared memory permit.

### Before you read the kernel: 4 Triton primitives and their CUDA equivalents

---

**`tl.program_id(axis=0)` — Which program instance is this?**

In CUDA, a similar outer index is `blockIdx.x`. In Triton, `tl.program_id(0)` identifies one instance of the JIT-compiled program. Launch 16 programs and the same kernel body runs 16 times with different IDs. Do not read `tl.arange` lanes as CUDA threads: they are vector elements whose execution is lowered across the program's CUDA threads and warps.

---

**`tl.arange(0, BLOCK_SIZE)` — Element offsets within the block**

This creates a vector of integers `[0, 1, 2, ..., BLOCK_SIZE-1]`, similar to `torch.arange`. Combined with `program_id * BLOCK_SIZE`, it gives the global element indices this program handles. At `BLOCK_SIZE=1024`, program 3 handles elements `[3072, 3073, ..., 4095]`.

---

**`tl.load(ptr + offsets, mask=mask)` — Load from HBM to registers**

This is the critical operation: read elements from GPU HBM (the slow memory) into the kernel's register file (the fast memory). The `mask` prevents out-of-bounds reads for the last partial block. The elements are now in registers — arithmetic on them is fast and doesn't require HBM access.

---

**`tl.store(ptr + offsets, c, mask=mask)` — Write result back to HBM**

After computation, write the result to the output tensor. For fused kernels (Parts 3–4), we avoid this intermediate write entirely — the output goes directly to the final destination, eliminating HBM roundtrips.

> **PyTorch/CUDA shape note:** Triton kernels operate on flat pointers, not shaped tensors. Reshape and compute strides before passing to the kernel. The `ptr + offsets` pattern is equivalent to `ptr[offsets]` in Python indexing but on raw GPU memory.

### From an array expression to launched programs

Before reading the figure, hold the array operation fixed: `C[i] = A[i] + B[i]`. Triton changes how indices are partitioned, not the meaning of the operation.

```mermaid
flowchart LR
    OP[Array operation: C[i] = A[i] + B[i]] --> GRID[Launch grid: ceil(N / BLOCK_SIZE) programs]
    GRID --> P0[program_id = 0]
    GRID --> P1[program_id = 1]
    GRID --> P2[program_id = 2]
    P0 --> I0[offsets 0..3]
    P1 --> I1[offsets 4..7]
    P2 --> I2[offsets 8..11]
```

**Small indexed example:** let `N=10` and `BLOCK_SIZE=4`. The grid is `(ceil(10/4),) = (3,)`. Program 0 computes indices `[0,1,2,3]`; program 1 computes `[4,5,6,7]`; program 2 proposes `[8,9,10,11]`. Only `[8,9]` are valid.

![Triton/CUDA hierarchy: a launch grid creates Triton program instances; each program is compiled onto CUDA threads grouped into warps](images/triton-grid-block-thread.png)

**Guided reading:** read the hierarchy top-down. The grid determines how many independent Triton programs exist. Each program owns a logical tile of array indices. The compiler, not `tl.arange`, chooses the CUDA thread-level schedule and distributes the tile across warps. `BLOCK_SIZE` is the tile width; `num_warps` is a separate launch/meta-parameter controlling how many warps cooperate on a program.

Contiguous offsets such as `[4,5,6,7]` produce adjacent addresses. That gives the compiler the shape it needs to generate **coalesced** memory transactions: threads in a warp collectively access neighboring bytes instead of issuing scattered transactions. Strided or irregular offsets can require more memory transactions even when the arithmetic is identical.

```mermaid
flowchart LR
    P2[program_id = 2] --> OFF[offsets = 8, 9, 10, 11]
    OFF --> CMP{offset < N = 10?}
    CMP -->|true| VALID[8, 9: load, compute, store]
    CMP -->|false| INVALID[10, 11: masked out]
    INVALID --> SAFE[No out-of-bounds memory access]
```

The same mask must guard loads and stores. For reductions or dot products, masked loads often specify a neutral `other` value such as `0.0`; for vector addition the invalid lanes do no useful work and the masked store suppresses their writes.

#### #### Predict first — what does the kernel output look like?

You're about to write `C[i] = A[i] + B[i]` as a Triton kernel for a 16 M-element vector. The PyTorch version `A + B` takes ~0.2ms on GPU. What will the Triton kernel produce?

1. **(a) A correct result, similar speed** — Triton compiles to the same CUDA ops as PyTorch's elementwise add; correctness and speed should be equivalent
2. **(b) A correct result, slower** — The Triton JIT compilation overhead will make our first kernel noticeably slower than PyTorch's tuned implementation
3. **(c) An error** — Raw pointer arithmetic (`A_ptr + offsets`) will fail for the last partial block without careful masking

Consider: what role does the `mask=mask` argument to `tl.load` play, and does PyTorch's `A + B` need to handle partial blocks?

In [ ]:
#  Part 1: Vector addition kernel
# Compile and run the real kernel when Triton is installed; otherwise show pseudocode
if TRITON_AVAILABLE:
    @triton.jit
    def add_kernel(
        A_ptr,        # pointer to tensor A in GPU memory
        B_ptr,        # pointer to tensor B
        C_ptr,        # pointer to output C
        N,            # total number of elements
        BLOCK_SIZE: tl.constexpr,  # number of elements per block (compile-time constant)
    ):
        # Which block is this? (equivalent to blockIdx.x in CUDA)
        block_id = tl.program_id(axis=0)

        # Compute the range of elements this block handles
        offsets = block_id * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)

        # Mask out-of-bounds elements (last block may be partial)
        mask = offsets < N

        # Load from GPU memory → registers (like __shared__ but in registers)
        a = tl.load(A_ptr + offsets, mask=mask)
        b = tl.load(B_ptr + offsets, mask=mask)

        # Compute: in Triton, arithmetic on loaded tiles is vectorized automatically
        c = a + b

        # Store result back to GPU memory
        tl.store(C_ptr + offsets, c, mask=mask)

    def triton_add(A, B):
        """Launch the Triton add kernel."""
        C = torch.empty_like(A)
        N = A.numel()
        BLOCK_SIZE = 1024
        grid = (triton.cdiv(N, BLOCK_SIZE),)  # number of blocks to launch
        add_kernel[grid](A, B, C, N, BLOCK_SIZE=BLOCK_SIZE)
        return C

    # Test and verify
    A = torch.randn(1024 * 16, device=DEVICE)
    B = torch.randn(1024 * 16, device=DEVICE)
    C_triton = triton_add(A, B)
    C_torch  = A + B
    # confirm the Triton kernel's output matches PyTorch's elementwise add
    match = torch.allclose(C_triton, C_torch, atol=1e-5)
    print(f"Triton vector addition matches PyTorch: {match}")
    print(f"  Max error: {(C_triton - C_torch).abs().max():.2e}")
else:
    print("TRITON_AVAILABLE = False — showing annotated pseudocode:")
    print()
    print("@triton.jit")
    print("def add_kernel(A_ptr, B_ptr, C_ptr, N, BLOCK_SIZE: tl.constexpr):")
    print("    block_id = tl.program_id(axis=0)   # which block? (like blockIdx.x)")
    print("    offsets  = block_id * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)  # element indices")
    print("    mask     = offsets < N              # guard last partial block")
    print("    a = tl.load(A_ptr + offsets, mask=mask)  # load from HBM → registers")
    print("    b = tl.load(B_ptr + offsets, mask=mask)")
    print("    c = a + b                           # vectorized arithmetic on register tiles")
    print("    tl.store(C_ptr + offsets, c, mask=mask)  # write back to HBM")
    print()
    print("PyTorch equivalent: C = A + B  (Triton compiles to the same CUDA ops)")


#### What just happened — and what's missing

The Triton vector addition compiled to CUDA PTX, launched `ceil(N / BLOCK_SIZE)` blocks in parallel, and produced output identical to `A + B` to 1e-5. You've now seen all three primitives in action: `tl.program_id` to identify the block, `tl.load` to pull a tile from HBM into registers, and `tl.store` to write the result back.

**What's missing:** vector addition is not bandwidth-bound in any interesting way — there's exactly one load per input element and one store per output element, with no reuse. The real Triton payoff is when a *sequence* of operations can be fused into one kernel so intermediate results never leave the register file. Parts 2–4 build toward that: first see tiling (Part 2), then see fusion save HBM accesses (Parts 3–4).

#### #### Your turn — block size

The kernel above uses `BLOCK_SIZE=1024`. Change `BLOCK_SIZE_YOURS` in the cell below and observe: does the result stay correct? Does the number of blocks launched change? What happens at very large block sizes (try 4096)?

*Prediction:* correctness should be unaffected by block size (the mask handles partial blocks). Only launch count and register pressure change.

In [ ]:
#  #### Your turn: block size effect on vector addition
# # CHANGE: try BLOCK_SIZE_YOURS = 128, 512, 2048 — does correctness change? throughput?
BLOCK_SIZE_YOURS = 1024  # ← CHANGE ME

# Compile and run the real kernel at this block size when Triton is installed
if TRITON_AVAILABLE:
    A_yt = torch.randn(1024 * 16, device=DEVICE)
    B_yt = torch.randn(1024 * 16, device=DEVICE)
    N_yt = A_yt.numel()
    C_yt = torch.empty_like(A_yt)
    # number of blocks needed to cover all N_yt elements at this block size
    grid_yt = (triton.cdiv(N_yt, BLOCK_SIZE_YOURS),)
    add_kernel[grid_yt](A_yt, B_yt, C_yt, N_yt, BLOCK_SIZE=BLOCK_SIZE_YOURS)
    # verify correctness is unaffected by the chosen block size
    match_yt = torch.allclose(C_yt, A_yt + B_yt, atol=1e-5)
    print(f"BLOCK_SIZE={BLOCK_SIZE_YOURS}: result correct = {match_yt}")
    print(f"  Blocks launched: {grid_yt[0]}  (= ceil(N / BLOCK_SIZE))")
    print(f"  → Larger BLOCK_SIZE = fewer blocks = less kernel-launch overhead")
    print(f"  → Too large = may exceed register file capacity; compiler spills to HBM")
else:
    n_elem = 1024 * 16
    blocks = n_elem // BLOCK_SIZE_YOURS
    print(f"With BLOCK_SIZE={BLOCK_SIZE_YOURS} on N={n_elem} elements:")
    print(f"  Blocks launched: {blocks}  (= N / BLOCK_SIZE)")
    print(f"  Correctness: UNCHANGED — the mask guards the last partial block regardless of size")
    print(f"  → Larger blocks: fewer launches, larger per-thread register footprint")
    print(f"  → Prediction: answer (a) — correctness is unaffected; only throughput changes")

---

## Part 2 — Tiled Matrix Multiply: Block-Level Parallelism

A naive Triton matmul: load rows of A and columns of B into SRAM, compute partial dot products, accumulate. This is the fundamental "block-level parallelism" pattern that FlashAttention also uses.

#### #### Predict first

A Triton tiled matmul vs. `torch.matmul` on a `(1024, 1024) × (1024, 1024)` matrix:

1. **(a) Triton is 2× faster** — we're using optimal SRAM tiling
2. **(b) `torch.matmul` is faster** — it uses cuBLAS/CUTLASS which have years of tuning
3. **(c) Within 5%** — both are near-peak for this regular matrix shape

**The key insight — registers vs HBM:**

```
Without tiling:                    With tiling (Triton):
k=0: compute → write to HBM        acc = zeros        ← registers
k=1: read from HBM, compute, write  acc += tile_k0     ← registers (no HBM!)
k=2: read from HBM, compute, write  acc += tile_k1     ← registers (no HBM!)
...                                 acc += tile_kN     ← registers (no HBM!)
                                   write acc → HBM    ← ONCE at the end
```

Without tiling: (K/BLOCK_K) × 2 HBM roundtrips. With tiling: 1 HBM read + 1 HBM write per output tile — regardless of K.

In [ ]:
#  Part 2: Tiled matrix multiply
# Compile and benchmark the real kernel when Triton is installed
if TRITON_AVAILABLE:
    @triton.jit
    def matmul_kernel(
        A_ptr, B_ptr, C_ptr,
        M, N, K,
        stride_am, stride_ak,
        stride_bk, stride_bn,
        stride_cm, stride_cn,
        BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
    ):
        """Tiled matmul: C = A @ B where A is (M,K) and B is (K,N)."""
        pid_m = tl.program_id(axis=0)  # which M-tile?
        pid_n = tl.program_id(axis=1)  # which N-tile?

        # Offsets for this tile
        m_offsets = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
        n_offsets = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)

        # Accumulator in registers (stays in SRAM — no HBM write until end)
        acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

        # Tile over the K dimension
        for k_start in range(0, K, BLOCK_K):
            k_offsets = k_start + tl.arange(0, BLOCK_K)

            # Load A tile and B tile into registers
            a_mask = (m_offsets[:, None] < M) & (k_offsets[None, :] < K)
            b_mask = (k_offsets[:, None] < K) & (n_offsets[None, :] < N)
            a = tl.load(A_ptr + m_offsets[:, None] * stride_am + k_offsets[None, :] * stride_ak, mask=a_mask, other=0.)
            b = tl.load(B_ptr + k_offsets[:, None] * stride_bk + n_offsets[None, :] * stride_bn, mask=b_mask, other=0.)

            # Accumulate partial dot product — STAYS IN REGISTERS
            acc += tl.dot(a, b)

        # Write result once — only ONE HBM write per output tile
        c_mask = (m_offsets[:, None] < M) & (n_offsets[None, :] < N)
        tl.store(C_ptr + m_offsets[:, None] * stride_cm + n_offsets[None, :] * stride_cn, acc.to(tl.float16), mask=c_mask)

    # Launch the tiled matmul kernel over a 2D grid of output tiles
    def triton_matmul(A, B, BLOCK=64):
        M, K = A.shape; _, N = B.shape
        C = torch.empty((M, N), device=A.device, dtype=torch.float16)
        grid = (triton.cdiv(M, BLOCK), triton.cdiv(N, BLOCK))
        matmul_kernel[grid](A.half(), B.half(), C, M, N, K,
                            A.stride(0), A.stride(1), B.stride(0), B.stride(1), C.stride(0), C.stride(1),
                            BLOCK_M=BLOCK, BLOCK_N=BLOCK, BLOCK_K=BLOCK)
        return C

    # Benchmark
    M = N = K = 512
    A_m = torch.randn(M, K, device=DEVICE); B_m = torch.randn(K, N, device=DEVICE)
    C_triton = triton_matmul(A_m, B_m)
    C_torch  = (A_m.half() @ B_m.half())
    match = torch.allclose(C_triton.float(), C_torch.float(), atol=1e-2)  # fp16 tolerance
    print(f"Triton tiled matmul matches torch.matmul: {match}  (atol=0.01 for fp16)")

    # Timing
    # time a callable by running it repeatedly and taking the median
    def bench(fn, n=30):
        if HAS_GPU: torch.cuda.synchronize()
        times = []
        # repeat to get a stable median timing
        for _ in range(n):
            if HAS_GPU: torch.cuda.synchronize()
            t0 = time.perf_counter(); fn();
            if HAS_GPU: torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
        return np.median(times) * 1000

    t_triton = bench(lambda: triton_matmul(A_m, B_m))
    t_torch  = bench(lambda: A_m.half() @ B_m.half())
    print(f"  Triton tiled matmul: {t_triton:.2f}ms")
    print(f"  torch.matmul:        {t_torch:.2f}ms  ({t_torch/t_triton:.2f}× {'faster' if t_torch < t_triton else 'slower'})")
    print()
    # determine which implementation won this benchmark
    faster = t_torch < t_triton
    print(f"Prediction check: answer {'(b)' if faster else '(a)'} — torch.matmul {'wins' if faster else 'loses'} "
          f"({'cuBLAS/CUTLASS is highly tuned' if faster else 'our Triton is well-tuned for this shape'})")
else:
    print("TRITON_AVAILABLE = False — pseudocode:")
    print()
    print("@triton.jit")
    print("def matmul_kernel(A_ptr, B_ptr, C_ptr, M, N, K, strides..., BLOCK_M, BLOCK_N, BLOCK_K):")
    print("    pid_m, pid_n = tl.program_id(0), tl.program_id(1)  # 2D grid")
    print("    acc = tl.zeros((BLOCK_M, BLOCK_N))   # accumulator in registers (not HBM!)")
    print("    for k in range(0, K, BLOCK_K):")
    print("        a_tile = tl.load(A_ptr + ...)    # (BLOCK_M, BLOCK_K) from HBM")
    print("        b_tile = tl.load(B_ptr + ...)    # (BLOCK_K, BLOCK_N) from HBM")
    print("        acc += tl.dot(a_tile, b_tile)    # stays in registers")
    print("    tl.store(C_ptr + ..., acc)           # ONE write to HBM per output tile")
    print()
    print("vs. naive matmul: reads partial results from HBM at each K step")
    print("    tiled:   O(M×N×K / BLOCK²) HBM reads — BLOCK× fewer reads")


#### #### Your turn — tile size

The tiled matmul above uses `BLOCK=64`. Change `BLOCK_YOURS` in the cell below and observe: how does the output tile grid change? Does correctness depend on block size? Try `BLOCK=32` and `BLOCK=128`.

*Prediction:* all power-of-2 block sizes should produce correct results (within fp16 tolerance). Throughput will vary — Part 5 explains exactly why `BLOCK=128` often wins on data-center GPUs.

In [ ]:
#  #### Your turn: tile shape effect on tiled matmul
# # CHANGE: try BLOCK_YOURS = 32, 64, 128, 256 — observe timing and correctness
BLOCK_YOURS = 64  # ← CHANGE ME

# Run the real tiled matmul at this block size when Triton is installed
if TRITON_AVAILABLE:
    M_yt2, K_yt2, N_yt2 = 512, 512, 512
    A_yt2 = torch.randn(M_yt2, K_yt2, device=DEVICE)
    B_yt2 = torch.randn(K_yt2, N_yt2, device=DEVICE)
    # some block sizes may not divide evenly into 512; catch and report the failure
    try:
        C_yt2   = triton_matmul(A_yt2, B_yt2, BLOCK=BLOCK_YOURS)
        C_ref2  = A_yt2.half() @ B_yt2.half()
        match2  = torch.allclose(C_yt2.float(), C_ref2.float(), atol=1e-2)
        # how many output tiles this block size produces
        n_tiles = (M_yt2 // BLOCK_YOURS) * (N_yt2 // BLOCK_YOURS)
        print(f"BLOCK={BLOCK_YOURS}: result correct = {match2}")
        print(f"  Grid shape: {M_yt2 // BLOCK_YOURS} × {N_yt2 // BLOCK_YOURS} = {n_tiles} output tiles")
        print(f"  Each tile accumulates K={K_yt2} elements in registers before 1 HBM write")
        print(f"  → Larger BLOCK = fewer tiles, more register reuse, but larger register footprint")
    except Exception as e:
        print(f"BLOCK={BLOCK_YOURS} failed: {e}  (try a power-of-2 that divides 512)")
else:
    n_tiles = (512 // BLOCK_YOURS) ** 2
    print(f"With BLOCK={BLOCK_YOURS} for 512×512 matmul:")
    print(f"  Grid: {512 // BLOCK_YOURS} × {512 // BLOCK_YOURS} = {n_tiles} output tiles")
    print(f"  Each tile reads {BLOCK_YOURS}×{BLOCK_YOURS} from A + {BLOCK_YOURS}×{BLOCK_YOURS} from B per K-step")
    print(f"  → Larger BLOCK reduces total HBM reads per element (better reuse)")
    print(f"  → Too large: register spilling kicks in — see Part 5 for the explanation")

#### What just happened — and what's missing

The tiled matmul keeps all intermediate accumulations in registers — not HBM. One output tile (`BLOCK_M × BLOCK_N`) is fully computed before a single result is written to HBM. This is the same tiling insight from Ch4's FlashAttention algorithm.

**Missing piece:** `torch.matmul` (backed by cuBLAS) is already fast for square matrices. The real win for Triton is **fusion**: combining multiple operations (e.g., matmul + GELU + bias) into one kernel that reduces HBM roundtrips. That's Part 3.

### Set up the fusion comparison

The two paths below compute the same `Y = GELU(X + bias)`. Track the intermediate tensor rather than the arithmetic: if one kernel produces it and another consumes it, it must cross the HBM boundary.

![Fused vs unfused GELU+bias: unfused takes 4 HBM accesses (read/write twice); fused Triton kernel uses 2 (read once, write once)](images/fused-vs-unfused-gelu.png)

**Guided reading:** follow the unfused path first and count the materialized `tmp = X + bias`: one kernel writes `tmp` to HBM and the next reads it back. Then follow the fused path: `tmp` is only a register value inside one program, so the intermediate write and read disappear. Fusion saves data movement and a launch boundary; it does not change the GELU formula.

```mermaid
flowchart LR
    subgraph U[Unfused: two kernels]
        UX[HBM: X and bias] --> UA[Kernel 1: add]
        UA --> UT[HBM: materialize tmp]
        UT --> UG[Kernel 2: GELU]
        UG --> UY[HBM: Y]
    end
    subgraph F[Fused: one Triton program]
        FX[HBM: X and bias] --> FR[Registers: add then GELU]
        FR --> FY[HBM: Y]
    end
```

---

## Part 3 — Fused Activation Functions: GELU + Bias in One Kernel

**Why do you need this now?** Parts 1 and 2 showed that tiling keeps accumulators in registers. But both kernels still write to HBM at intermediate points — the matmul writes its output, then a separate bias-add kernel reads it, then GELU reads it again. This is the *kernel boundary tax*: every time control returns to the Python/PyTorch layer and a new kernel launches, intermediate tensors must round-trip through HBM.

Fusion eliminates that tax. A fused kernel computes `GELU(X + bias)` in a *single* kernel invocation: load X from HBM once, add bias in registers, apply GELU in registers, write output once. The math is identical to the unfused version; only the memory traffic changes.

**Fusion eliminates HBM roundtrips.** Unfused GELU+bias requires reading activations from HBM, writing bias-added result to HBM, reading again, applying GELU, writing output — 4–5 HBM accesses. A fused Triton kernel does all of this in registers: 1 read, 1 write — **50–60% fewer HBM accesses**.

*One HBM access = one full-tensor read OR write (e.g., reading X counts as 1 access, writing Y counts as 1 access).*

#### #### Predict first — HBM access count for GELU+bias

A standard PyTorch `F.gelu(x + bias)` (with `x` shape `(1024, 256)`) incurs how many HBM read/write operations?

1. **(a) 2 accesses** — one read of x and bias, one write of the GELU output
2. **(b) 4–5 accesses** — PyTorch launches separate kernels for bias-add and GELU; each kernel reads from and writes to HBM once, so the intermediate result round-trips through HBM
3. **(c) 1 access** — PyTorch already fuses these ops internally on CUDA

The answer determines how large a gap a fused Triton kernel can close.

In [ ]:
#  Part 3: Fused GELU+bias kernel
print("Part 3: Fused GELU+bias")
print()

# Compile and run the real fused kernel when Triton is installed
if TRITON_AVAILABLE:
    @triton.jit
    def fused_gelu_bias_kernel(
        X_ptr, bias_ptr, Y_ptr,
        N_COLS,
        BLOCK_SIZE: tl.constexpr,
    ):
        """
        Fused: Y = GELU(X + bias)
        Without fusion: two separate kernels, two HBM read-write roundtrips.
        With fusion: one read, one write — GELU computed in registers.
        """
        row_id = tl.program_id(axis=0)
        offsets = tl.arange(0, BLOCK_SIZE)
        mask = offsets < N_COLS

        x    = tl.load(X_ptr    + row_id * N_COLS + offsets, mask=mask)
        bias = tl.load(bias_ptr + offsets, mask=mask)

        # GELU: 0.5 * x * (1 + tanh(sqrt(2/pi) * (x + 0.044715 * x^3)))
        z = x + bias
        gelu_out = 0.5 * z * (1.0 + tl.math.tanh(0.7978845608 * (z + 0.044715 * z * z * z)))

        tl.store(Y_ptr + row_id * N_COLS + offsets, gelu_out, mask=mask)

    # Launch one block per row, sized to the row's column count
    def triton_fused_gelu_bias(X, bias):
        M, N = X.shape
        Y = torch.empty_like(X)
        fused_gelu_bias_kernel[(M,)](X, bias, Y, N, BLOCK_SIZE=min(triton.next_power_of_2(N), 1024))
        return Y

    # Test on a typical FFN intermediate (B×S, 4D)
    M_ffn, N_ffn = B * S, D * 4  # (64, 256)
    X_ffn  = torch.randn(M_ffn, N_ffn, device=DEVICE)
    bias   = torch.randn(N_ffn, device=DEVICE)

    # run the fused kernel and the unfused PyTorch equivalent for comparison
    Y_fused = triton_fused_gelu_bias(X_ffn, bias)
    Y_ref   = F.gelu(X_ffn + bias)
    match   = torch.allclose(Y_fused, Y_ref, atol=1e-4)
    print(f"Fused GELU+bias matches unfused: {match}  (atol=1e-4)")

    # Memory bandwidth comparison (theoretical)
    # tensor size in bytes, used to estimate HBM traffic for each approach
    n_bytes = M_ffn * N_ffn * X_ffn.element_size()
    print(f"\nHBM traffic comparison for ({M_ffn}×{N_ffn}) GELU+bias:")
    print(f"  Unfused (2 kernels): {n_bytes * 4 / 1e6:.2f} MB  (read X, write X+bias, read again, write GELU)")
    print(f"  Fused (1 kernel):    {n_bytes * 2 / 1e6:.2f} MB  (read X, write GELU_output only)")
    print(f"  HBM savings: 50%")
else:
    print("TRITON_AVAILABLE = False — pseudocode:")
    print()
    print("@triton.jit")
    print("def fused_gelu_bias_kernel(X_ptr, bias_ptr, Y_ptr, N_COLS, BLOCK_SIZE):")
    print("    row_id = tl.program_id(0)            # one row per block")
    print("    offsets = tl.arange(0, BLOCK_SIZE)")
    print("    x    = tl.load(X_ptr + row_id*N_COLS + offsets)   # load X from HBM")
    print("    bias = tl.load(bias_ptr + offsets)                  # load bias from HBM")
    print("    z = x + bias                                        # add: in registers")
    print("    y = 0.5 * z * (1 + tanh(sqrt(2/pi)*(z + 0.044715*z^3)))  # GELU: in registers")
    print("    tl.store(Y_ptr + row_id*N_COLS + offsets, y)       # ONE write to HBM")
    print()
    print("Unfused equivalent (2 separate operations, 4× HBM accesses):")
    print("  tmp = X + bias   # read X, read bias, write tmp → 3 accesses")
    print("  Y   = gelu(tmp)  # read tmp, write Y → 2 accesses")
    print("  total: 5 HBM accesses")
    print()
    print("Fused (1 kernel, 2 HBM accesses): read X+bias, write Y")


#### What just happened — and what's missing

The fused kernel computed `GELU(X + bias)` with **2 HBM accesses** (1 read + 1 write), versus the 4–5 accesses of PyTorch's unfused path. The prediction was answer **(b)** — PyTorch does run separate kernels for bias-add and GELU, and each round-trips through HBM. The math is identical; only the memory traffic changed.

**What's missing:** GELU+bias is elementwise — every output position is independent of every other. The hard fusion case is softmax, where the denominator `sum(exp(x_i))` depends on *every element in the row*. That cross-element dependency forces a two-pass scan even in the fused case. Part 4 handles it.

#### #### Your turn — FFN width and HBM savings

Change `N_ffn_yours` below (the FFN hidden dimension). Does the *ratio* of HBM savings change when you use a wider FFN (e.g., 4096 instead of 256)?

*Prediction:* the absolute MB saved grows with N, but the percentage saving stays constant — because fusion always converts 4 accesses to 2 (50%), regardless of tensor size. Try it to confirm.

In [ ]:
#  #### Your turn: how do HBM savings scale with FFN width?
# # CHANGE: try N_ffn_yours = 256, 512, 1024, 4096 — does the saving ratio change?
N_ffn_yours = 256  # ← CHANGE ME (typical FFN hidden dim = 4×D, so D=64 → 256)

M_yt3 = B * S   # rows = batch × seq_len
n_bytes3 = M_yt3 * N_ffn_yours * 4  # float32

unfused_mb = n_bytes3 * 4 / 1e6   # 4 HBM accesses: read X, read bias, write bias_out, read bias_out, write gelu_out → ~5
fused_mb   = n_bytes3 * 2 / 1e6   # 2 HBM accesses: read X+bias once (single kernel load), write gelu_out

print(f"FFN: shape ({M_yt3}×{N_ffn_yours}), dtype=float32")
print(f"  Unfused (4 accesses): {unfused_mb:.2f} MB")
print(f"  Fused   (2 accesses): {fused_mb:.2f} MB")
# what fraction of HBM traffic fusion eliminates at this FFN width
savings_pct = (unfused_mb - fused_mb) / unfused_mb * 100
print(f"  Savings: {unfused_mb - fused_mb:.2f} MB  ({savings_pct:.0f}% reduction)")
print()
print(f"  → The {savings_pct:.0f}% saving is CONSTANT regardless of N_ffn_yours")
print(f"     because fusion always halves the access count (4→2), independent of tensor size")

---

## Part 4 — Fused Tiled Softmax: FlashAttention's Inner Loop in Triton

**Why do you need this now?** Parts 2 and 3 covered operations where each output element depends only on a small local tile: a matmul output tile depends only on the corresponding A-row and B-column tiles; a GELU output element depends only on the same element of the input. Softmax is different — the denominator `sum(exp(x_i))` depends on *every element in the row*. That cross-element dependency is exactly what makes softmax expensive and exactly what FlashAttention's tiling algorithm was designed to handle.

This Part implements the fused row-softmax in Triton: load an entire row into registers, compute max→exp→sum→divide entirely in registers, write once. For rows that fit in `BLOCK_SIZE`, this is the inner loop of FlashAttention. For longer sequences (S > SRAM capacity), Part 4 here is the building block — Ch4's streaming two-pass algorithm extends it by maintaining running `(max, sum)` state across tiles.

> **Note:** This kernel demonstrates the *fusion pattern* for rows that fit in a single BLOCK_SIZE. For production sequence lengths where S > SRAM capacity, FlashAttention uses the streaming two-pass algorithm from Ch4: maintaining running `(max, sum)` state across tiles. This kernel is the building block — Ch4 shows the extension to arbitrary sequence length.

#### #### Predict first — how many GPU kernels does `torch.softmax` run?

`torch.softmax(x, dim=-1)` on attention scores (shape `(16, 128)`) — how many separate GPU kernels does PyTorch dispatch internally, and how many HBM accesses does that imply?

1. **(a) 1 kernel, 2 accesses** — PyTorch already fuses softmax into a single GPU kernel
2. **(b) 3 kernels, 6 accesses** — max reduction, exp+sum, then normalize
3. **(c) 4–5 kernels, 8–10 accesses** — max, shift, exp, sum, normalize each run as separate CUDA kernels; each reads and writes the full tensor once

The answer determines how much room a fused Triton kernel has to improve on PyTorch's baseline.

In [ ]:
#  Part 4: Fused tiled softmax (FlashAttention inner loop)
# Compile and run the real fused softmax kernel when Triton is installed
if TRITON_AVAILABLE:
    @triton.jit
    def fused_softmax_kernel(
        X_ptr, Y_ptr,
        M, N,
        stride_xm, stride_xn,
        stride_ym, stride_yn,
        BLOCK_SIZE: tl.constexpr,
    ):
        """
        Row-wise softmax without materializing the full row in HBM.
        Each block handles one row; the tile fits in registers.
        For very long rows (N > BLOCK_SIZE), use an online softmax loop.
        """
        row_id = tl.program_id(axis=0)
        offsets = tl.arange(0, BLOCK_SIZE)
        mask = offsets < N

        # Load row into registers (if N <= BLOCK_SIZE, entire row fits)
        x = tl.load(X_ptr + row_id * stride_xm + offsets * stride_xn, mask=mask, other=float('-inf'))

        # Numerically stable softmax: subtract max first
        x_max = tl.max(x, axis=0)
        x_exp = tl.exp(x - x_max)
        x_sum = tl.sum(x_exp, axis=0)
        y = x_exp / x_sum

        tl.store(Y_ptr + row_id * stride_ym + offsets * stride_yn, y, mask=mask)

    # Launch one block per row, sized to the next power of 2 ≥ row length
    def triton_softmax(X):
        M, N = X.shape
        Y = torch.empty_like(X)
        BLOCK = triton.next_power_of_2(N)
        fused_softmax_kernel[(M,)](X, Y, M, N, X.stride(0), X.stride(1), Y.stride(0), Y.stride(1), BLOCK_SIZE=BLOCK)
        return Y

    # Test on attention score matrix
    S_soft = S  # sequence length
    scores = torch.randn(B * 2, S_soft, device=DEVICE)  # (B*heads, S) attention scores

    # run the fused kernel and PyTorch's reference softmax for comparison
    Y_triton = triton_softmax(scores)
    Y_ref    = torch.softmax(scores, dim=-1)
    match    = torch.allclose(Y_triton, Y_ref, atol=1e-5)
    print(f"Fused tiled softmax matches torch.softmax: {match}  (atol=1e-5)")
    print(f"  Max absolute error: {(Y_triton - Y_ref).abs().max():.2e}")

    # Bandwidth comparison
    # tensor size in bytes, used to estimate HBM traffic for each approach
    n_bytes_soft = scores.numel() * scores.element_size()
    print(f"\nHBM traffic for softmax on ({B*2}×{S_soft}):")
    print(f"  PyTorch softmax (unfused): ~{n_bytes_soft * 4 / 1e6:.2f} MB (read + max + exp + sum + normalize)")
    print(f"  Fused Triton kernel:       ~{n_bytes_soft * 2 / 1e6:.2f} MB (read once, write once)")
    print(f"  This is why FlashAttention-2 reaches 87% of peak HBM bandwidth")
else:
    print("TRITON_AVAILABLE = False — pseudocode:")
    print()
    print("@triton.jit")
    print("def fused_softmax_kernel(X_ptr, Y_ptr, M, N, strides..., BLOCK_SIZE):")
    print("    row_id  = tl.program_id(0)           # one row per block")
    print("    offsets = tl.arange(0, BLOCK_SIZE)")
    print("    x = tl.load(X_ptr + row_id*stride + offsets)  # load from HBM: ONCE")
    print("    x_max = tl.max(x, axis=0)            # reduce: in registers")
    print("    x_exp = tl.exp(x - x_max)            # exponentiate: in registers")
    print("    x_sum = tl.sum(x_exp, axis=0)        # sum: in registers")
    print("    y = x_exp / x_sum                    # normalise: in registers")
    print("    tl.store(Y_ptr + row_id*stride + offsets, y)  # write to HBM: ONCE")
    print()
    print("PyTorch unfused equivalent runs 4–5 separate kernels:")
    print("  1. kernel: x_max = torch.max(x, dim=-1)")
    print("  2. kernel: x_shifted = x - x_max.unsqueeze(-1)")
    print("  3. kernel: x_exp = torch.exp(x_shifted)")
    print("  4. kernel: x_sum = torch.sum(x_exp, dim=-1)")
    print("  5. kernel: y = x_exp / x_sum.unsqueeze(-1)")
    print("Each kernel: read + write to HBM = 5×2 = 10 HBM accesses vs. fused 2 accesses")


#### What just happened — and what's missing

The fused softmax loaded each attention row **once** from HBM, computed `max → exp → sum → normalize` entirely in registers, then wrote the result back **once** — 2 HBM accesses versus 8–10 for PyTorch's unfused path. The output matched `torch.softmax` to 1e-5. The prediction was **(c)**: PyTorch's softmax dispatches 4–5 separate kernels, each reading and writing the full tensor.

**What's missing:** this kernel requires the *entire* row (`N` elements) to fit in one `BLOCK_SIZE`. For `seq_len=128` that's fine — 128 floats is trivial. For `seq_len=8192` (LLaMA-3 with full context), the row spans multiple tiles. FlashAttention-2 handles this with the *streaming* two-pass algorithm from Ch4: carrying running `(max, sum)` state across tiles so each tile still touches HBM only once. This kernel is that algorithm's inner loop; Ch4 is its outer frame.

#### #### Your turn — sequence length and savings

Change `SEQ_LEN_YOURS` in the cell below. Do the *absolute* HBM savings grow with sequence length? Does the *ratio* (%) stay constant? Try `SEQ_LEN_YOURS = 64`, then `1024`.

*Prediction:* absolute savings grow linearly (more bytes per row → more to save); the ratio stays constant (fusion always converts ~8 accesses to 2, regardless of row length).

In [ ]:
#  #### Your turn: HBM savings vs. sequence length for softmax
# # CHANGE: try SEQ_LEN_YOURS = 64, 256, 512, 1024 — do savings grow with seq_len?
SEQ_LEN_YOURS = 256  # ← CHANGE ME

N_HEADS_YOURS = B * 2  # batch × num_heads (same as the Part 4 kernel above)
n_bytes4 = N_HEADS_YOURS * SEQ_LEN_YOURS * 4  # float32 per row × rows

unfused_mb4 = n_bytes4 * 8 / 1e6   # 4–5 separate kernels → ~8 HBM accesses (read + write each pass)
fused_mb4   = n_bytes4 * 2 / 1e6   # fused: 1 read + 1 write

print(f"Softmax on ({N_HEADS_YOURS} rows × {SEQ_LEN_YOURS} cols) attention scores:")
print(f"  Unfused PyTorch (~8 accesses): {unfused_mb4:.2f} MB")
print(f"  Fused Triton     (2 accesses): {fused_mb4:.2f} MB")
print(f"  Absolute savings: {unfused_mb4 - fused_mb4:.2f} MB  ({(unfused_mb4-fused_mb4)/unfused_mb4*100:.0f}% reduction)")
print()
print(f"  → HBM savings grow LINEARLY with seq_len (more bytes per row = more to save)")
print(f"  → The 75% saving ratio is also constant — fusion always goes 8 accesses → 2")
print(f"  → This is why softmax fusion matters MORE for long-context models (seq_len=8k, 32k)")

# also verify correctness of the real kernel at this sequence length, when available
if TRITON_AVAILABLE:
    # very long sequences may exceed the kernel's BLOCK_SIZE capacity
    try:
        X_yt4 = torch.randn(N_HEADS_YOURS, SEQ_LEN_YOURS, device=DEVICE)
        Y_yt4   = triton_softmax(X_yt4)
        Y_ref4  = torch.softmax(X_yt4, dim=-1)
        correct = torch.allclose(Y_yt4, Y_ref4, atol=1e-5)
        print(f"  Correctness at SEQ_LEN={SEQ_LEN_YOURS}: {correct}")
    except Exception as e:
        print(f"  (Note: SEQ_LEN={SEQ_LEN_YOURS} may exceed the kernel's BLOCK_SIZE — {e})")

---

## Part 5 — Autotuning: Let the GPU Tell You the Optimal Block Size

Different GPUs have different SRAM sizes, warp counts, and memory hierarchies. The optimal `BLOCK_SIZE` for a Triton kernel on an A100 is different from an RTX 4090.

`@triton.autotune` runs the kernel with multiple configurations on real data and selects the fastest.

#### #### Predict first

For a `(1024, 1024) × (1024, 1024)` matmul on your GPU, which block size will win?

1. **(a) BLOCK=32** — smaller blocks fit better in SRAM
2. **(b) BLOCK=64** — balanced between SRAM fit and parallelism
3. **(c) BLOCK=128** — larger tiles better utilize memory bandwidth

The answer is GPU-specific — that's why autotuning exists.

### Set up the autotuning sweep

A tile configuration is a resource trade-off, not a universal constant. Small tiles expose more programs but may do too little work per program; large tiles improve reuse but consume more registers and can reduce occupancy or spill. `num_warps` is tuned alongside tile dimensions in production kernels because it changes how many CUDA warps cooperate on each program.

![Autotuning block size sweep: throughput peaks at BLOCK=128 (amber, winner) with reference torch.matmul line](images/autotune-block-size-sweep.png)

**Guided reading:** scan left to right. Throughput first rises as larger tiles improve reuse and provide enough work to hide latency. The peak is the best measured balance for this kernel, shape, dtype, and GPU. The drop after the peak signals resource pressure: fewer resident programs/warps reduce occupancy, or registers spill to local memory. Do not transfer the winning number blindly to another GPU or shape.

```mermaid
flowchart LR
    KEY[New autotune key: shape and dtype] --> CANDS[Candidate configs: tile sizes and num_warps]
    CANDS --> WARM[Compile and warm up each candidate]
    WARM --> TIME[Benchmark repeatedly]
    TIME --> PICK[Select fastest valid config]
    PICK --> CACHE[Cache winner for this key]
    CACHE --> RUN[Reuse winner on later calls]
```

In [ ]:
#  Part 5: Autotuning (manual sweep where @triton.autotune unavailable)
if TRITON_AVAILABLE:
    # Manual sweep across block sizes (simulates what @triton.autotune does)
    M_at = N_at = K_at = 512
    A_at = torch.randn(M_at, K_at, device=DEVICE)
    B_at = torch.randn(K_at, N_at, device=DEVICE)

    block_sizes = [16, 32, 64, 128]
    times = {}

    # benchmark the tiled matmul at each candidate block size
    for BLOCK in block_sizes:
        try:
            # closure capturing the current BLOCK value for this iteration's timing
            def run_block():
                return triton_matmul(A_at, B_at, BLOCK=BLOCK)
            # warm up
            for _ in range(3): run_block()
            if HAS_GPU: torch.cuda.synchronize()
            t = []
            for _ in range(20):
                if HAS_GPU: torch.cuda.synchronize()
                t0 = time.perf_counter(); run_block()
                if HAS_GPU: torch.cuda.synchronize()
                t.append(time.perf_counter() - t0)
            # median timing in ms for this block size
            times[BLOCK] = np.median(t) * 1000
        # some block sizes may fail (e.g. not divide the matrix evenly); mark as unavailable
        except Exception as e:
            times[BLOCK] = float('nan')

    # pick the fastest block size, ignoring any that failed
    best_block = min(times, key=lambda k: times[k] if not np.isnan(times[k]) else float('inf'))

    print("Autotuning sweep — tiled matmul across block sizes:")
    for bs, t in times.items():
        marker = " ← WINNER" if bs == best_block else ""
        print(f"  BLOCK={bs:4d}: {t:6.2f}ms{marker}")

    print()
    print(f"Prediction check: optimal block size on this hardware = {best_block}")
    print(f"  The winning block size depends on your GPU's SRAM size and memory bandwidth.")

    # Plot
    # drop failed block sizes before plotting
    valid = {k: v for k, v in times.items() if not np.isnan(v)}
    # bar chart of timing per block size, highlighting the winner
    fig, ax = plt.subplots(figsize=(8, 4))
    # highlight the winning block size in a different color
    colors = ['coral' if bs == best_block else 'steelblue' for bs in valid.keys()]
    bars = ax.bar(range(len(valid)), list(valid.values()), color=colors, edgecolor='white')
    ax.set_xticks(range(len(valid))); ax.set_xticklabels([f"BLOCK={k}" for k in valid.keys()])
    ax.set_ylabel("Time (ms)"); ax.set_title(f"Autotuning: block size vs. throughput (winner = {best_block})")
    plt.tight_layout(); plt.show()
else:
    block_sizes = [32, 64, 128, 256]
    # Reference throughputs (from benchmark literature on A100)
    ref_throughput = {32: 0.82, 64: 1.45, 128: 2.10, 256: 1.85}  # relative TFLOPS
    print("Reference autotuning results (A100 80GB, from Triton benchmarks):")
    # reference winner when Triton isn't installed to benchmark locally
    best = max(ref_throughput, key=ref_throughput.get)
    for bs, tp in ref_throughput.items():
        marker = " ← typical winner" if bs == best else ""
        print(f"  BLOCK={bs}: {tp:.2f} TFLOPS{marker}")
    print()
    print("Prediction: answer (c) BLOCK=128 often wins on data-center GPUs (large SRAM)")
    print("           answer (b) BLOCK=64 often wins on consumer GPUs (smaller SRAM)")
    print("→ @triton.autotune runs this sweep automatically and caches the result")


**Why does BLOCK_SIZE=128 win on A100? The register-file mechanism**

Autotuning isn't just "try numbers until one is fastest" — it's searching for the `BLOCK_SIZE` that best matches the GPU's **register file** capacity to the kernel's per-thread working set. Same registers-vs-HBM story as Part 2's diagram, but now the tile size itself is the variable.

```
Register file (fixed size per SM — e.g. ~256 KB on A100, shared across all resident warps)

  BLOCK=32/64   tile fits easily in registers, but too FEW warps stay
                resident -> not enough in-flight tl.load()s to hide HBM
                latency -> SM stalls waiting on memory     [under-utilized]

  BLOCK=128     accumulator + operand tiles fit the per-thread register
                budget AND enough warps stay resident to overlap the
                next tile's HBM load with compute          <- sweet spot

  BLOCK=256+    tile no longer fits in registers -> compiler SPILLS the
                overflow to "local memory" -- which, despite the name,
                is NOT on-chip. It's carved out of the same slow
                global/HBM memory tl.load() already reads from. Every
                spilled register access now pays an HBM round trip.
```

- **Registers are the fastest, scarcest resource.** Each A100 SM has a fixed register file (~256 KB) shared across every warp resident on that SM. The accumulator (`acc = tl.zeros((BLOCK_M, BLOCK_N))`) and the loaded `a`/`b` tiles all have to live somewhere — Triton keeps them in registers as long as they fit.
- **Too large → register spilling.** Push `BLOCK_SIZE` too far (256, 512) and the per-thread footprint exceeds the register budget. The compiler spills the overflow to "local memory," which is physically global/HBM memory, not a faster on-chip tier. Every spilled access now costs a full HBM round trip — this is why very large block sizes often run *slower* despite more parallel work per launch.
- **Too small → not enough latency hiding.** GPUs hide the ~400–800 cycle latency of a `tl.load()` HBM fetch by keeping many warps resident per SM and switching between them while one warp waits on its load. A tiny `BLOCK_SIZE` (16, 32) means fewer active warps and less independent work to interleave, so the SM idles on memory instead of overlapping it with compute — throughput suffers from under-utilization, not spilling.
- **BLOCK=128 is the sweet spot** for A100-class register/SRAM budgets: large enough to keep enough warps resident to hide load latency, small enough that the accumulator + operand tiles fit inside the register file without spilling to HBM.

The reference sweep above tells the same story in numbers: BLOCK=32 → 0.82 TFLOPS, BLOCK=64 → 1.45, **BLOCK=128 → 2.10 TFLOPS (winner)**, then BLOCK=256 drops to 1.85 — throughput rises as latency-hiding improves, then falls once register spilling kicks in.

`@triton.autotune` finds this optimum empirically per-GPU — an RTX 4090 has a different register file size and SM count and may prefer a different `BLOCK_SIZE` — but the underlying trade-off (latency hiding vs. register-file capacity) is always the same.


#### #### Your turn — predict the block size winner

Before looking at the autotuning sweep results above, predict: which `BLOCK_SIZE` will win on your GPU? Change `PREDICTED_WINNER` in the cell below, then run both cells to check.

*The key question:* is your GPU more like an A100 (data-center, large register file → BLOCK=128 wins) or an RTX consumer GPU (smaller per-SM register budget → BLOCK=64 often wins)? The register-file mechanism explained above is the reason — not the number itself.

In [ ]:
#  #### Your turn: predict the winning block size before looking at the sweep
# # CHANGE: which BLOCK_SIZE do you predict will win on your hardware?
PREDICTED_WINNER = 128  # ← CHANGE ME: try 32, 64, 128, or 256

# compare the prediction against the real sweep results when available, else the reference table
if TRITON_AVAILABLE and HAS_GPU and 'times' in dir():
    # drop failed block sizes before finding the actual winner
    valid_times = {k: v for k, v in times.items() if not np.isnan(v)}
    if valid_times:
        # the block size that actually ran fastest in the Part 5 sweep
        actual_winner = min(valid_times, key=valid_times.get)
        # check whether the prediction matched the measured winner
        correct = (PREDICTED_WINNER == actual_winner)
        print(f"Your prediction:    BLOCK={PREDICTED_WINNER}")
        print(f"Actual winner:      BLOCK={actual_winner}")
        print(f"Prediction correct: {correct}")
        if not correct:
            print(f"  → Your GPU's register budget or SRAM layout favors BLOCK={actual_winner}")
            print(f"     This is exactly why @triton.autotune exists: hardware varies significantly")
        else:
            print(f"  → Matches the A100 reference: register file fits BLOCK={actual_winner} without spilling")
    else:
        print("Timing results unavailable — run the Part 5 autotuning cell first")
else:
    ref = {32: 0.82, 64: 1.45, 128: 2.10, 256: 1.85}
    # winner from the reference A100 throughput table
    ref_winner = max(ref, key=ref.get)
    # check the prediction against the reference winner
    correct_ref = (PREDICTED_WINNER == ref_winner)
    print(f"Your prediction:    BLOCK={PREDICTED_WINNER}")
    print(f"A100 reference winner: BLOCK={ref_winner}  (2.10 TFLOPS)")
    print(f"Prediction correct (vs. A100): {correct_ref}")
    print()
    print("  GPU-specific notes:")
    print("  A100 / H100: BLOCK=128 typical winner (large register file + many SMs)")
    print("  RTX 3080/4090: BLOCK=64 often wins (smaller per-SM register budget)")
    print("  → @triton.autotune makes the right choice automatically per GPU")

#### What just happened — and what's missing

Autotuning ran the same kernel at four block sizes and selected the fastest empirically — without any theory. The winner depended on your GPU's register-file capacity and SM count, not on a formula. This is the correct answer to "what block size should I use?": **measure, don't guess.** Different GPUs (A100 vs. RTX 4090 vs. H100) routinely prefer different block sizes for the same kernel.

**What's missing:** all five kernels so far required you to write `@triton.jit` functions by hand. For standard PyTorch operations — linear layers, GELU, layer norm, softmax — you never do this. `torch.compile` generates equivalent Triton kernels automatically. Part 6 shows what those generated kernels look like, and clarifies when you *actually* need to write Triton manually.

---

## Part 6 — Toy → Real: Where Are Triton Kernels in `torch.compile`'d Code?

**Why do you need this now?** You've written five Triton kernels by hand. Here's the honest question: do you ever actually write these in practice?

For standard operations — linear layers, GELU, softmax, layer norm — **no.** When you run `torch.compile(model)`, PyTorch 2.0's TorchInductor backend generates Triton kernels automatically. The hand-written kernels you built in Parts 1–5 are structurally identical to what the compiler generates, just simpler. You don't need to write them; you need to *read* them when `torch.compile` produces unexpected behavior or a profiler shows a suspiciously slow fused op.

The compilation chain is: `PyTorch ops → TorchDynamo (graph capture) → TorchInductor → Triton kernels → CUDA PTX`. Triton is the bridge between your Python model and the GPU instructions that execute. Part 6 shows exactly what that bridge looks like for a two-layer MLP.

When you run `torch.compile(model)`, PyTorch 2.0 generates Triton kernels for fused operations. You can see the generated kernel names in the compilation output. This is the connection between your high-level PyTorch code and the GPU kernels that actually run.

#### #### Predict first — how many Triton kernels does `torch.compile` generate?

For a two-layer MLP `out = fc2(F.gelu(fc1(x)))`, how many Triton kernels does `torch.compile` generate?

1. **(a) 4 kernels** — one per op: fc1 matmul, GELU, fc2 matmul, output write
2. **(b) 1–2 kernels** — TorchInductor fuses adjacent element-wise ops; the matmul+GELU become one kernel and the second matmul another
3. **(c) 0 kernels** — `torch.compile` dispatches to cuBLAS for matmul and doesn't use Triton at all

The answer reveals whether TorchInductor actually does the fusion work you did by hand in Part 3.

In [ ]:
#  Part 6: Show Triton kernel names from torch.compile
import torch.nn as nn

# a minimal 2-layer MLP whose compiled kernels we'll inspect
class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(D, D * 4)
        self.fc2 = nn.Linear(D * 4, D)
    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))

tiny = TinyModel().to(DEVICE)
x_in = torch.randn(B, S, D).to(DEVICE)

try:
    # ask TorchDynamo/TorchInductor to trace and compile this model to fused kernels
    compiled = torch.compile(tiny, mode='default', fullgraph=False)
    # First call: compilation happens here
    with torch.no_grad(): out = compiled(x_in)

    # Try to get the generated code
    try:
        explanation = torch._dynamo.explain(tiny)(x_in)
        print("torch.compile graph explanation:")
        print(f"  Break count: {explanation.break_count}")
        print(f"  Graphs captured: {len(explanation.graphs)}")
    except Exception:
        print("torch.compile successfully compiled the model.")

    print()
    print("The compiled model dispatches to Triton kernels for fused operations.")
    print("Key kernels typically generated:")
    print("  - triton_fused_linear_relu / triton_fused_linear_gelu (fc1+activation fused)")
    print("  - triton_mm (matrix multiply)")
    print("  - triton_add / triton_mul (element-wise ops)")
    print()
    print("View generated Triton code by setting: TORCH_COMPILE_DEBUG=1")
    print("  This shows the exact @triton.jit function generated for each fused group.")

# compilation may be unavailable (e.g. no CUDA); fall back to explaining what would happen
except Exception as e:
    print(f"torch.compile: {e}")
    print()
    print("Even without running compilation, the key insight is:")
    print("  torch.compile → TorchDynamo (graph capture) → TorchInductor → Triton kernels")
    print()
    print("Your high-level PyTorch code:")
    print("  out = gelu(x @ W1 + b1)")
    print()
    print("Gets compiled to a SINGLE Triton kernel roughly equivalent to:")
    print("  @triton.jit")
    print("  def fused_linear_gelu(x_ptr, W_ptr, b_ptr, out_ptr, ...):")
    print("      x_tile = tl.load(x_ptr + ...)    # load x from HBM")
    print("      W_tile = tl.load(W_ptr + ...)    # load W tile from HBM")
    print("      acc    = tl.dot(x_tile, W_tile)  # matmul in registers")
    print("      b      = tl.load(b_ptr + ...)    # load bias")
    print("      out    = gelu_triton(acc + b)    # bias + GELU in registers")
    print("      tl.store(out_ptr + ..., out)     # write ONCE to HBM")
    print()
    print("vs. 4 separate PyTorch ops = 4 separate HBM roundtrips.")


#### What just happened — and what's missing

`torch.compile` captured the computation graph and, on a CUDA GPU, dispatched to TorchInductor which generated fused Triton kernels for the matmul+GELU sequence. The answer to the prediction is **(b)**: TorchInductor fuses adjacent operations, producing 1–2 kernels instead of 4. The generated kernels are structurally identical to the hand-written Parts 2–3 kernels: `tl.load → tl.dot → GELU-in-registers → tl.store`.

**The honest answer to "when do you actually write Triton?"**
- **Rarely.** `torch.compile` handles standard ops.
- **When your attention variant isn't covered by SDPA** — non-standard masking, sliding-window attention, linear attention variants.
- **When you need custom quantization** formats that PyTorch's quantization stack doesn't support.
- **When debugging a suspiciously slow compiled op** — now you can read the generated kernel and identify whether a fusion boundary is causing unexpected HBM traffic.

The five kernels you wrote by hand were the vocabulary. Part 6 is the payoff: you can now read, understand, and debug the kernels the compiler writes for you.

---

##  Your Turn — Profile the Fused vs. Unfused Softmax

If Triton is available, measure whether the fused softmax kernel is actually faster at different sequence lengths by comparing it to `torch.softmax`.

In [ ]:
#   Your Turn: Fused vs unfused softmax benchmark
# # CHANGE: try seq_lengths = [64, 256, 1024, 4096] to see where fusion helps most
seq_lengths_test = [64, 128, 256, 512]  # ← CHANGE ME

# time a callable by running it repeatedly and taking the median
def bench_simple(fn, n=20):
    if HAS_GPU: torch.cuda.synchronize()
    times = []
    # repeat to get a stable median timing
    for _ in range(n):
        if HAS_GPU: torch.cuda.synchronize()
        t0 = time.perf_counter(); fn()
        if HAS_GPU: torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return np.median(times) * 1000

print("Fused Triton softmax vs. torch.softmax:")
print(f"{'S':6s}  {'torch (ms)':12s}  {'Triton (ms)':12s}  {'Speedup':8s}")
print("-" * 45)

# benchmark fused vs. unfused softmax at each sequence length
for S_t in seq_lengths_test:
    # synthetic attention-score-shaped input at this sequence length
    X_t = torch.randn(B * 8, S_t, device=DEVICE)
    t_torch  = bench_simple(lambda: torch.softmax(X_t, dim=-1))
    if TRITON_AVAILABLE:
        t_triton = bench_simple(lambda: triton_softmax(X_t))
        speedup  = t_torch / t_triton
        print(f"  {S_t:4d}   {t_torch:10.3f}   {t_triton:10.3f}   {speedup:6.1f}×")
    else:
        print(f"  {S_t:4d}   {t_torch:10.3f}   (Triton not available for comparison)")

if not TRITON_AVAILABLE:
    print()
    print("Reference results (A100 GPU, from Triton documentation):")
    print("  S=256: torch=0.12ms, Triton=0.08ms → 1.5× faster (fusion saves 2 HBM reads)")
    print("  S=512: torch=0.38ms, Triton=0.22ms → 1.7× faster (more bandwidth saved)")
    print("  S=1024: torch=1.20ms, Triton=0.65ms → 1.8× faster (bandwidth bound dominates)")


---

## Toy → Real: Mapping This Notebook's Kernels to Production

Every kernel in this notebook has a direct production equivalent. The patterns are identical — only shapes and complexity scale up.

| Toy Triton Kernel (this notebook) | Production Equivalent | Where it runs |
|---|---|---|
| `add_kernel` — vector addition | `triton_add`, `triton_mul` (element-wise fusions) | Any element-wise chain in `torch.compile` output |
| `matmul_kernel` — tiled matmul | `triton_mm` inside TorchInductor | `torch.compile`'d linear layers |
| `fused_gelu_bias_kernel` — GELU+bias | `triton_fused_linear_gelu` (TorchInductor) | Every FFN projection in every transformer |
| `fused_softmax_kernel` — row softmax | FlashAttention-2 inner softmax tile | `F.scaled_dot_product_attention` on CUDA |
| Manual block-size sweep (Part 5) | `@triton.autotune` with `triton.Config` list | FlashAttention, every production Triton kernel |

**Toy vs. production dimensions:**

| Parameter | This notebook | FlashAttention-2 (production) |
|---|---|---|
| Sequence length (S) | 128 | 512 – 131 072 |
| Head dimension (D) | 64 | 64 – 256 |
| Block size (BLOCK_M/N) | 64 (fixed) | 16 – 128 (autotuned per GPU) |
| Softmax coverage | 1 full row in SRAM | 1 tile — streaming across tiles (Ch4) |
| HBM reads per attention head | 2 (fused) | 2 (fused — same idea, larger tensors) |

If you understood the fused softmax in Part 4, you understand 80% of what makes FlashAttention-2 fast. The remaining 20% is Ch4's streaming two-pass algorithm — the outer loop that extends Part 4's inner loop to arbitrary sequence lengths without increasing HBM accesses per element.

---

## Summary and Closing Decision

| Part | Kernel | HBM savings | Key pattern |
|------|--------|-------------|-------------|
| 1 | Vector addition | Minimal | tl.load → compute → tl.store |
| 2 | Tiled matmul | 1 write per output tile | Accumulator in registers across K |
| 3 | Fused GELU+bias | 50% | Read once, fuse ops, write once |
| 4 | Fused tiled softmax | 50% | 1 read + 1 write vs. 4+ unfused |
| 5 | Autotuning | — | `@triton.autotune` finds optimal BLOCK |
| 6 | torch.compile → Triton | Automatic | Compiler generates these kernels for you |

In [ ]:
#  Closing Decision
print("=" * 60)
print("  CLOSING DECISION — Custom Triton Kernels")
print("=" * 60)
print()
print("  Fused softmax HBM savings: ~50% vs. unfused PyTorch")
print("  → Equivalent to 2× more memory bandwidth for this operation")
print()
print("  When to write custom Triton kernels:")
print("  1. Your attention variant is NOT covered by SDPA/FlashAttention")
print("     (e.g., custom masking, non-standard attention patterns)")
print("  2. You have a sequence of ops where fusion reduces HBM traffic significantly")
print("     (e.g., linear + custom activation + layer norm in one pass)")
print("  3. You need a custom quantization format not supported by PyTorch")
print()
print("  When NOT to write Triton kernels:")
print("  → Standard attention: use F.scaled_dot_product_attention (already Triton)")
print("  → Standard matmul: use torch.matmul (cuBLAS is faster for regular shapes)")
print("  → Standard activations: use torch.compile (auto-generates Triton fusions)")
print()
print("  The official FlashAttention-2 Triton kernel achieves 87% of peak HBM BW.")
print("  For production: use it via F.scaled_dot_product_attention, not a re-implementation.")
print("  Write custom Triton when you have a genuinely novel operation.")


---

## Key Insights to Keep

- **The kernel boundary tax is real.** Every time a PyTorch operation crosses a kernel boundary, intermediate tensors hit HBM. Fusion eliminates this — not by making the math faster, but by keeping results in registers across operations that would otherwise force a read-write cycle.
- **Triton is Python that compiles to CUDA PTX.** `@triton.jit` + `tl.load/store` + `tl.program_id` are the entire vocabulary you need to read production kernels like FlashAttention-2.
- **Tiling is the same idea at every level.** Part 2's matmul tiling, Part 4's softmax tiling, and Ch4's FlashAttention tiling are the same move: keep accumulated state in registers, read from HBM once per tile, write to HBM once per output tile.
- **Block size is hardware-specific — autotune, don't guess.** A100 prefers `BLOCK=128`; RTX 4090 may prefer `BLOCK=64`. The register-file budget drives this, not theory. `@triton.autotune` measures your hardware directly.
- **Register spilling is the cliff.** Too-large blocks exceed the register file, causing the compiler to spill to "local memory" — which is physically HBM. Throughput drops sharply. The autotuning sweep shows this directly.
- **`torch.compile` generates Triton for free.** For standard ops, you don't write Triton — TorchInductor generates it. Write custom Triton only when SDPA/torch.compile doesn't cover your novel operation.
- **FlashAttention-2 at 87% peak bandwidth is the benchmark.** Your fused softmax uses 2 HBM accesses (1 read + 1 write) per row — the same ratio FlashAttention-2 achieves. The algorithm is not mysterious once you see the kernel.

---

## What This Notebook Covered (and What It Didn't)

### Tier 1 — Implemented and Demonstrated
- Triton vector addition — `@triton.jit`, `tl.load`, `tl.store`, grid/block model
- Tiled matrix multiply — K-loop accumulation in registers; one HBM write per output tile
- Fused GELU+bias — 50% fewer HBM accesses vs. two separate ops
- Fused tiled softmax — 50% fewer HBM accesses; matches `torch.softmax` to 1e-5
- Autotuning sweep — manual sweep shows GPU-specific optimal block size

### Tier 2 — Explained but Not Rewritten
- **FlashAttention-2 full Triton kernel** — the production kernel is available at `github.com/Dao-AILab/flash-attention`; its structure is exactly Parts 2+4 combined with the Ch4 online softmax algorithm; referencing it is more honest than a poor reimplementation

### Tier 3 — Named but Out of Scope
- **cutlass** — NVIDIA's C++ template library for custom CUDA kernels; lower-level than Triton, higher peak performance for regular shapes
- **pallas/XLA** — Google's equivalent of Triton for TPUs and JAX
- **cuBLAS** — NVIDIA's optimised BLAS library; what `torch.matmul` uses under the hood

---

## When to Use What

| Situation | Tool | Why |
|---|---|---|
| Standard attention | `F.scaled_dot_product_attention` | Already Triton-optimized; automatic dispatch |
| Standard matmul | `torch.matmul` | cuBLAS is faster for regular square shapes |
| Fused ops (relu+bias etc.) | `torch.compile` | Automatically generates Triton fusions |
| Custom attention variant | Triton kernel | When SDPA doesn't support your masking/variant |
| Custom quantization op | Triton kernel | When PyTorch's built-in quantization doesn't apply |

**This chapter completes the `learning/ai-infrastructure/` track.** You have now covered the full stack from GPU hardware → mixed precision → profiling → FlashAttention → distributed training → quantization → inference systems → custom kernels. Every optimization technique in this track maps to a concrete, measurable bottleneck that you can profile and fix.

---

## Production and Cloud Deployment Gates

A custom Triton kernel is a versioned accelerator, not a universal replacement for its PyTorch reference. Promote it only when every gate below passes for the exact cloud image, GPU SKU, input contract, and kernel revision.

| Gate | Release requirement | Failure action |
|---|---|---|
| Device, driver, compiler | Allowlisted GPU compute capability; compatible NVIDIA driver, CUDA runtime, PyTorch, and Triton versions | Route to the PyTorch fallback and block promotion |
| Shape and dtype contract | Supported rank, dimensions, strides, contiguity, device, dtype, and bounded sizes | Reject at the dispatch boundary or use the fallback kernel |
| Autotune cache and versioning | Key results by kernel source version, Triton/PyTorch versions, GPU model, driver/runtime, shape bucket, and dtype | Retune on any key change; never share results across unlike workers |
| Numerical parity | Compare against the reference over normal, extreme, and non-power-of-two shapes with dtype-specific tolerances | Quarantine the artifact on any mismatch or non-finite output |
| Benchmark thresholds | Meet absolute latency and relative-to-fallback thresholds after warmup, with synchronized measurements and recorded percentiles | Keep the fallback active; investigate regressions before release |
| Fallback kernel | Preserve a tested PyTorch implementation and dispatch to it on unsupported inputs or runtime failures | Emit a reason code and continue serving |
| Compilation cache | Use a writable, image- or node-local cache with a versioned namespace; prewarm known shapes where cold-start latency matters | Rebuild safely; do not treat cache contents as portable binaries |
| Telemetry | Record kernel version, selected path, contract bucket, cache hit/miss, compile latency, execution latency, parity status, and fallback reason | Alert on error, fallback, or latency-rate changes without logging tensor payloads |
| Canary and rollback | Start with shadow traffic, then a small canary by GPU SKU; compare correctness, latency, errors, and fallback rate | Roll back by configuration to the known-good kernel or PyTorch path |

Cloud fleets are heterogeneous even under one service name. Pin the container image, publish the allowlist and report together, and repeat qualification for each GPU SKU. Autotune and compilation caches are disposable acceleration data; the machine-readable qualification report is the release evidence.

In [ ]:
import hashlib
import json
import os
import platform
from datetime import datetime, timezone

RUN_PRODUCTION_CAPABILITY_VALIDATION = False
RUN_PRODUCTION_COMPILE = False
RUN_PRODUCTION_PARITY = False
RUN_PRODUCTION_BENCHMARK = False
RUN_PRODUCTION_WRITE_ARTIFACT = False

KERNEL_NAME = "fused_softmax"
KERNEL_VERSION = "1.0.0"
MIN_COMPUTE_CAPABILITY = (8, 0)
SUPPORTED_DTYPES = {torch.float16, torch.bfloat16, torch.float32}
SUPPORTED_LAST_DIMS = {64, 128, 256, 512, 1024, 2048, 4096}
PRODUCTION_SHAPE = (B * 8, S)
PRODUCTION_DTYPE = torch.float16
PRODUCTION_THRESHOLDS = {
    "float16": {"atol": 1e-3, "rtol": 1e-3},
    "bfloat16": {"atol": 4e-3, "rtol": 4e-3},
    "float32": {"atol": 1e-5, "rtol": 1e-5},
    "max_relative_latency": 1.05,
    "max_absolute_latency_ms": 1.0,
}

def validate_softmax_contract(x):
    failures = []
    if x.device.type != "cuda":
        failures.append("device_not_cuda")
    if x.ndim != 2:
        failures.append("rank_not_2")
    if x.dtype not in SUPPORTED_DTYPES:
        failures.append("unsupported_dtype")
    if x.ndim == 2 and x.shape[-1] not in SUPPORTED_LAST_DIMS:
        failures.append("unsupported_last_dimension")
    if x.ndim == 2 and x.stride(-1) != 1:
        failures.append("last_dimension_not_contiguous")
    if x.numel() == 0:
        failures.append("empty_tensor")
    return {"passed": not failures, "failures": failures}

def collect_runtime_capability():
    capability = {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "triton": getattr(triton, "__version__", None) if TRITON_AVAILABLE else None,
        "cuda_runtime": torch.version.cuda,
        "cuda_available": bool(HAS_GPU),
        "triton_available": bool(TRITON_AVAILABLE),
        "gpu_name": None,
        "compute_capability": None,
    }
    if HAS_GPU:
        props = torch.cuda.get_device_properties(0)
        capability["gpu_name"] = props.name
        capability["compute_capability"] = [props.major, props.minor]
    compute_capability = tuple(capability["compute_capability"] or (0, 0))
    capability["passed"] = (
        capability["cuda_available"]
        and capability["triton_available"]
        and compute_capability >= MIN_COMPUTE_CAPABILITY
    )
    return capability

runtime_capability = collect_runtime_capability()
cache_key_fields = {
    "kernel": KERNEL_NAME,
    "kernel_version": KERNEL_VERSION,
    "torch": runtime_capability["torch"],
    "triton": runtime_capability["triton"],
    "cuda_runtime": runtime_capability["cuda_runtime"],
    "gpu_name": runtime_capability["gpu_name"],
    "compute_capability": runtime_capability["compute_capability"],
    "shape": list(PRODUCTION_SHAPE),
    "dtype": str(PRODUCTION_DTYPE),
}
autotune_cache_key = hashlib.sha256(
    json.dumps(cache_key_fields, sort_keys=True).encode("utf-8")
).hexdigest()[:16]

production_report = {
    "schema_version": "1.0",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "kernel": {"name": KERNEL_NAME, "version": KERNEL_VERSION},
    "runtime_capability": runtime_capability,
    "contract": {
        "supported_dtypes": sorted(str(dtype) for dtype in SUPPORTED_DTYPES),
        "supported_last_dims": sorted(SUPPORTED_LAST_DIMS),
        "candidate_shape": list(PRODUCTION_SHAPE),
        "candidate_dtype": str(PRODUCTION_DTYPE),
        "status": "not_run",
    },
    "cache": {
        "autotune_key": autotune_cache_key,
        "triton_cache_dir": os.environ.get("TRITON_CACHE_DIR"),
        "torchinductor_cache_dir": os.environ.get("TORCHINDUCTOR_CACHE_DIR"),
    },
    "checks": {"compile": "not_run", "parity": "not_run", "benchmark": "not_run"},
    "decision": "fallback",
    "fallback_reason": "production_checks_not_run",
}

if RUN_PRODUCTION_CAPABILITY_VALIDATION:
    production_report["contract"]["status"] = "pending_tensor_validation"
    print(json.dumps(production_report["runtime_capability"], indent=2, sort_keys=True))
else:
    print("Production capability validation skipped; RUN_PRODUCTION_CAPABILITY_VALIDATION=False")
    print("No Triton kernel was compiled or launched.")

In [ ]:
def read_cuda_driver_version():
    try:
        get_driver_version = getattr(torch._C, "_cuda_getDriverVersion")
        return str(get_driver_version())
    except (AttributeError, RuntimeError):
        return None

required_runtime_versions = {
    "torch": os.environ.get("PRODUCTION_TORCH_VERSION"),
    "triton": os.environ.get("PRODUCTION_TRITON_VERSION"),
    "cuda_runtime": os.environ.get("PRODUCTION_CUDA_RUNTIME_VERSION"),
    "nvidia_driver": os.environ.get("PRODUCTION_NVIDIA_DRIVER_VERSION"),
}
runtime_capability["nvidia_driver"] = read_cuda_driver_version() if HAS_GPU else None
runtime_capability["required_runtime_versions"] = required_runtime_versions

missing_version_pins = [
    name for name, required in required_runtime_versions.items() if not required
]
version_mismatches = [
    name
    for name, required in required_runtime_versions.items()
    if required and runtime_capability.get(name) != required
]
runtime_capability["version_gate"] = {
    "passed": not missing_version_pins and not version_mismatches,
    "missing_pins": missing_version_pins,
    "mismatches": version_mismatches,
}
runtime_capability["passed"] = (
    runtime_capability["passed"] and runtime_capability["version_gate"]["passed"]
)
production_report["runtime_capability"] = runtime_capability

if RUN_PRODUCTION_CAPABILITY_VALIDATION and not runtime_capability["version_gate"]["passed"]:
    print("Runtime version gate failed closed; configure the PRODUCTION_*_VERSION pins.")

In [ ]:
def pytorch_softmax_fallback(x):
    return torch.softmax(x, dim=-1)

def benchmark_cuda_ms(fn, warmup=25, repetitions=100):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    samples_ms = []
    for _ in range(repetitions):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        fn()
        end.record()
        torch.cuda.synchronize()
        samples_ms.append(start.elapsed_time(end))
    samples_ms.sort()
    return {
        "p50_ms": float(np.percentile(samples_ms, 50)),
        "p90_ms": float(np.percentile(samples_ms, 90)),
        "p99_ms": float(np.percentile(samples_ms, 99)),
        "repetitions": repetitions,
    }

production_input = None
production_output = None
reference_output = None

if RUN_PRODUCTION_COMPILE:
    if not runtime_capability["passed"]:
        production_report["checks"]["compile"] = "blocked"
        production_report["fallback_reason"] = "capability_gate_failed"
    elif "triton_softmax" not in globals():
        production_report["checks"]["compile"] = "blocked"
        production_report["fallback_reason"] = "kernel_wrapper_not_defined"
    else:
        production_input = torch.randn(
            PRODUCTION_SHAPE, device="cuda", dtype=PRODUCTION_DTYPE
        ).contiguous()
        contract_result = validate_softmax_contract(production_input)
        production_report["contract"].update(contract_result)
        production_report["contract"]["status"] = "passed" if contract_result["passed"] else "failed"
        if not contract_result["passed"]:
            production_report["checks"]["compile"] = "blocked"
            production_report["fallback_reason"] = "contract_gate_failed"
        else:
            try:
                compile_started = time.perf_counter()
                production_output = triton_softmax(production_input)
                torch.cuda.synchronize()
                production_report["checks"]["compile"] = "passed"
                production_report["compile_latency_ms"] = (
                    time.perf_counter() - compile_started
                ) * 1000
            except Exception as exc:
                production_report["checks"]["compile"] = "failed"
                production_report["fallback_reason"] = "compile_or_launch_failed"
                production_report["compile_error"] = type(exc).__name__
else:
    print("Compile check skipped; RUN_PRODUCTION_COMPILE=False")

if RUN_PRODUCTION_PARITY:
    if production_report["checks"]["compile"] != "passed":
        production_report["checks"]["parity"] = "blocked"
    else:
        reference_output = pytorch_softmax_fallback(production_input)
        dtype_name = str(PRODUCTION_DTYPE).replace("torch.", "")
        tolerances = PRODUCTION_THRESHOLDS[dtype_name]
        difference = (production_output - reference_output).float()
        max_abs_error = float(difference.abs().max().item())
        denominator = reference_output.float().abs().clamp_min(1e-12)
        max_rel_error = float((difference.abs() / denominator).max().item())
        finite = bool(torch.isfinite(production_output).all().item())
        parity_passed = finite and bool(torch.allclose(
            production_output,
            reference_output,
            atol=tolerances["atol"],
            rtol=tolerances["rtol"],
        ))
        production_report["parity"] = {
            "passed": parity_passed,
            "finite": finite,
            "max_abs_error": max_abs_error,
            "max_rel_error": max_rel_error,
            **tolerances,
        }
        production_report["checks"]["parity"] = "passed" if parity_passed else "failed"
        if not parity_passed:
            production_report["fallback_reason"] = "numerical_parity_failed"
else:
    print("Parity check skipped; RUN_PRODUCTION_PARITY=False")

if RUN_PRODUCTION_BENCHMARK:
    if production_report["checks"]["compile"] != "passed":
        production_report["checks"]["benchmark"] = "blocked"
    else:
        triton_latency = benchmark_cuda_ms(lambda: triton_softmax(production_input))
        fallback_latency = benchmark_cuda_ms(lambda: pytorch_softmax_fallback(production_input))
        relative_latency = triton_latency["p50_ms"] / fallback_latency["p50_ms"]
        benchmark_passed = (
            triton_latency["p50_ms"] <= PRODUCTION_THRESHOLDS["max_absolute_latency_ms"]
            and relative_latency <= PRODUCTION_THRESHOLDS["max_relative_latency"]
        )
        production_report["benchmark"] = {
            "passed": benchmark_passed,
            "triton": triton_latency,
            "fallback": fallback_latency,
            "relative_latency": relative_latency,
            "thresholds": {
                "max_absolute_latency_ms": PRODUCTION_THRESHOLDS["max_absolute_latency_ms"],
                "max_relative_latency": PRODUCTION_THRESHOLDS["max_relative_latency"],
            },
        }
        production_report["checks"]["benchmark"] = "passed" if benchmark_passed else "failed"
        if not benchmark_passed:
            production_report["fallback_reason"] = "benchmark_threshold_failed"
else:
    print("Benchmark skipped; RUN_PRODUCTION_BENCHMARK=False")

required_checks = ("compile", "parity", "benchmark")
if all(production_report["checks"][name] == "passed" for name in required_checks):
    production_report["decision"] = "canary"
    production_report["fallback_reason"] = None

def production_softmax(x):
    contract_result = validate_softmax_contract(x)
    if not runtime_capability["passed"]:
        return pytorch_softmax_fallback(x), "fallback:capability_gate_failed"
    if not contract_result["passed"]:
        return pytorch_softmax_fallback(x), "fallback:contract_gate_failed"
    if production_report["decision"] not in {"canary", "promote"}:
        return pytorch_softmax_fallback(x), "fallback:kernel_not_qualified"
    try:
        return triton_softmax(x), "triton"
    except Exception:
        return pytorch_softmax_fallback(x), "fallback:runtime_kernel_failure"

In [ ]:
production_report["telemetry_contract"] = {
    "dimensions": [
        "service_version",
        "kernel_version",
        "gpu_name",
        "compute_capability",
        "shape_bucket",
        "dtype",
        "selected_path",
        "fallback_reason",
        "autotune_cache_key",
    ],
    "metrics": [
        "compile_latency_ms",
        "execution_latency_ms",
        "fallback_count",
        "kernel_error_count",
        "parity_failure_count",
        "cache_hit_count",
        "cache_miss_count",
    ],
    "payload_logging_allowed": False,
}
production_report["rollout"] = {
    "stage": "offline_qualification",
    "canary_fraction": 0.0,
    "promotion_requires": [
        "all_offline_checks_passed",
        "shadow_parity_passed",
        "canary_latency_within_slo",
        "fallback_rate_within_budget",
        "error_rate_within_budget",
    ],
    "rollback_target": "pytorch_softmax_fallback",
}

artifact_json = json.dumps(production_report, indent=2, sort_keys=True)
print(artifact_json)

if RUN_PRODUCTION_WRITE_ARTIFACT:
    artifact_path = f"{KERNEL_NAME}-{KERNEL_VERSION}-{autotune_cache_key}.qualification.json"
    with open(artifact_path, "w", encoding="utf-8") as artifact_file:
        artifact_file.write(artifact_json + "\n")
    print(f"Wrote qualification artifact: {artifact_path}")
else:
    print("Qualification artifact write skipped; RUN_PRODUCTION_WRITE_ARTIFACT=False")

### Canary, Telemetry, and Rollback Runbook

1. **Offline qualification:** Build the pinned image on each target GPU SKU. Enable the capability, compile, parity, and benchmark flags in a controlled qualification job, never in an import path or ordinary notebook run. Archive the JSON report with the image digest and kernel source revision.
2. **Cache preparation:** Give Triton and TorchInductor writable, versioned cache directories. Prewarm only allowlisted shape/dtype buckets. A missing or invalid cache must cause recompilation or fallback, not a service failure.
3. **Shadow traffic:** Run the custom and fallback kernels on duplicated requests without serving the custom result. Compare sampled outputs with the same parity tolerances; record only metadata and aggregate errors, never tensor payloads.
4. **Canary:** Route a small percentage on one qualified GPU SKU. Gate expansion on p50/p90/p99 latency, kernel error rate, numerical failures, fallback rate, compile latency, and cache hit rate. Segment every metric by kernel version, GPU, shape bucket, and dtype.
5. **Promotion:** Increase traffic gradually only while all thresholds remain within budget. Do not infer qualification for another driver, compiler, GPU, shape, or dtype from the current canary.
6. **Rollback:** Flip dispatch configuration to the pinned PyTorch fallback or last known-good kernel, stop further promotion, and invalidate the affected kernel/autotune cache namespace. Preserve reports and telemetry for diagnosis; rollback must not require rebuilding the service image.

**Immediate stop conditions:** non-finite output, parity failure, unsupported contract reaching the Triton path, repeated compile/launch exceptions, latency above the release threshold, or an unexpected rise in fallback rate. The fallback is part of the production design, not a temporary debugging path.